In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
df = pd.read_csv('../data/dataset.csv')

df['Disease'] = df['Disease'].str.strip()
df[symptom_cols] = df[symptom_cols].apply(lambda col: col.str.strip())

print("shape:", df.shape)
# print("\nFirst 3 rows:")
df.head(3)

shape: (4920, 18)


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(df.columns.tolist())
print("Total disease:", df['Disease'].nunique())
print("\nAll disease:")
for d in sorted(df['Disease'].unique()):
    print(" ->", d)

['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']
Total disease: 41

All disease:
 -> (vertigo) Paroymsal  Positional Vertigo
 -> AIDS
 -> Acne
 -> Alcoholic hepatitis
 -> Allergy
 -> Arthritis
 -> Bronchial Asthma
 -> Cervical spondylosis
 -> Chicken pox
 -> Chronic cholestasis
 -> Common Cold
 -> Dengue
 -> Diabetes
 -> Dimorphic hemmorhoids(piles)
 -> Drug Reaction
 -> Fungal infection
 -> GERD
 -> Gastroenteritis
 -> Heart attack
 -> Hepatitis B
 -> Hepatitis C
 -> Hepatitis D
 -> Hepatitis E
 -> Hypertension
 -> Hyperthyroidism
 -> Hypoglycemia
 -> Hypothyroidism
 -> Impetigo
 -> Jaundice
 -> Malaria
 -> Migraine
 -> Osteoarthristis
 -> Paralysis (brain hemorrhage)
 -> Peptic ulcer diseae
 -> Pneumonia
 -> Psoriasis
 -> Tuberculosis
 -> Typhoid
 -> Urinary tract infection
 -> Varicose veins
 -

In [12]:
# Get all unique symptoms across all symptom columns 

symptom_cols = ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5',
                'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10',
                'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15',
                'Symptom_16', 'Symptom_17']

# Clean whitespace
df[symptom_cols] = df[symptom_cols].apply(lambda col: col.str.strip())

# Get all unique symptoms
all_symptoms = set()
for col in symptom_cols:
    all_symptoms.update(df[col].dropna().unique())

all_symptoms = sorted(all_symptoms)
print("✅ Total unique symptoms:", len(all_symptoms))

✅ Total unique symptoms: 131


In [13]:
# Create binary Matrix

def encode_symptoms(row):
    result = {symptom: 0 for symptom in all_symptoms}
    for col in symptom_cols:
        val = row[col]
        if pd.notna(val):
            val = str(val).strip()
            if val in result:
                result[val] = 1
    return result  # ← this must be OUTSIDE the for loop

binary_df = df.apply(encode_symptoms, axis=1, result_type='expand')
binary_df['Disease'] = df['Disease'].str.strip().values

# Make sure all symptom values are numeric
symptom_columns = [c for c in binary_df.columns if c != 'Disease']  # ← columns not column
binary_df[symptom_columns] = binary_df[symptom_columns].astype(int)

print("✅ Shape:", binary_df.shape)
print("✅ Dtype sample:", binary_df[symptom_columns[:3]].dtypes)
binary_df.head(3)

✅ Shape: (4920, 132)
✅ Dtype sample: abdominal_pain           int64
abnormal_menstruation    int64
acidity                  int64
dtype: object


,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin,Disease
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [14]:
X = binary_df.drop('Disease', axis=1)
y = binary_df['Disease']

print("Features (symptoms):", X.shape[1])
print("Samples:", X.shape[0])

Features (symptoms): 131
Samples: 4920


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# model = RandomForestClassifier(n_estimators=100, random_state=42)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
  #  min_samples_split=2,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, predictions))


✅ Accuracy: 1.0


In [17]:
# Test with some real symptoms

test_cases = [
    ['fever', 'headache', 'nausea'],
    ['itching', 'skin_rash', 'nodal_skin_eruptions'],
    ['chest_pain', 'breathlessness', 'sweating'],
]

for symptoms in test_cases:
    test_input = dict.fromkeys(all_symptoms, 0)
    for s in symptoms:
        if s in test_input:
            test_input[s] = 1
    
    input_df = pd.DataFrame([test_input])
    probabilities = model.predict_proba(input_df)[0]
    top3_idx = np.argsort(probabilities)[-3:][::-1]  # ← was [-3] missing the :
    top3 = [(model.classes_[i], round(probabilities[i]*100, 2)) for i in top3_idx]  # ← was top2_idx
    
    print(f"\n🤒 Symptoms: {symptoms}")
    print("🔬 Top 3 predictions:")
    for disease, conf in top3:
        print(f"   → {disease}: {conf}%")



🤒 Symptoms: ['fever', 'headache', 'nausea']
🔬 Top 3 predictions:
   → Heart attack: 3.43%
   → Malaria: 3.31%
   → Chronic cholestasis: 3.26%

🤒 Symptoms: ['itching', 'skin_rash', 'nodal_skin_eruptions']
🔬 Top 3 predictions:
   → Fungal infection: 15.48%
   → Heart attack: 3.1%
   → Jaundice: 2.87%

🤒 Symptoms: ['chest_pain', 'breathlessness', 'sweating']
🔬 Top 3 predictions:
   → Heart attack: 7.43%
   → Bronchial Asthma: 3.39%
   → GERD: 3.16%


In [18]:
desc_df = pd.read_csv('../data/symptom_Description.csv')
prec_df = pd.read_csv('../data/symptom_precaution.csv')


# clean column names and values
desc_df.columns = desc_df.columns.str.strip()
prec_df.columns = prec_df.columns.str.strip()
desc_df['Disease'] = desc_df['Disease'].str.strip()
prec_df['Disease'] = prec_df['Disease'].str.strip()


print("✅ Descriptions loaded:", desc_df.shape)
print("✅ Precaution loaded:", prec_df.shape)
print("\nDescription columns:", desc_df.columns.tolist())
print("Precaution columns:", prec_df.columns.tolist())


✅ Descriptions loaded: (41, 2)
✅ Precaution loaded: (41, 5)

Description columns: ['Disease', 'Description']
Precaution columns: ['Disease', 'Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4']


In [19]:
def get_disease_info(disease_name):
    disease_name = disease_name.strip()


    #Get description
    desc_row = desc_df[desc_df['Disease'] == disease_name]
    description = desc_row['Description'].values[0] if len(desc_row) > 0 else "No Description available."


    #Get Precaution
    prec_row = prec_df[prec_df['Disease'] == disease_name]
    if len(prec_row) > 0:
        precautions = [
            prec_row['Precaution_1'].values[0],
            prec_row['Precaution_2'].values[0],
            prec_row['Precaution_3'].values[0],
            prec_row['Precaution_4'].values[0],
        ]

        precautions = [p for p in precautions if pd.notna(p)]
    else:
        precautions = ["Consult a doctor immediately."]

    return description, precautions


In [22]:
test_disease = "Fungal infection"
desc, precs  = get_disease_info(test_disease)

print(f"🦠 Disease: {test_disease}")
print(f"\n📖 Description:\n{desc}")
print(f"\n⚠️ Precautions:")

for i, p in enumerate(precs, 1):
    print(f" {i}. {p}")

🦠 Disease: Fungal infection

📖 Description:
In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune system to handle. Fungi can live in the air, soil, water, and plants. There are also some fungi that live naturally in the human body. Like many microbes, there are helpful fungi and harmful fungi.

⚠️ Precautions:
 1. bath twice
 2. use detol or neem in bathing water
 3. keep infected area dry
 4. use clean cloths


In [1]:
import joblib
import json

model = joblib.load('../models/disease_model.pkl')
all_symptoms = joblib.load('../models/symptoms_list.pkl')

# Save symptoms as JSON (small file, safe for git)
with open('../backend/symptoms.json', 'w') as f:
    json.dump(all_symptoms, f)

print("✅ Symptoms saved as JSON!")

✅ Symptoms saved as JSON!
